In [1]:
import uuid
import logging
from base64 import b64encode
from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO)
logging.getLogger(__name__).setLevel(logging.DEBUG)
logging.getLogger("httpx").setLevel(logging.WARNING)
logger = logging.getLogger(__name__)
load_dotenv()

True

## Evaluation pipeline

### PREPARE DATA

In [2]:
from evals import Dataset

dataset_name = "test"
ds_custom = Dataset(dataset_name)
ds_baseline = Dataset(dataset_name)

data_custom = ds_custom.load_dataset()
data_baseline = ds_baseline.load_dataset()

if not data_custom or "sessions" not in data_custom:
    print("Ingen sessions funnet")
    exit()

ds_custom.assign_session_attachments()
ds_baseline.assign_session_attachments_baseline()

total_attachments = sum(len(s.get("attachments", [])) for s in data_custom["sessions"])
logger.info(f"Done — {len(data_custom['sessions'])} sessions, {total_attachments} attachments assigned in total")

INFO:pikepdf._core:pikepdf C++ to Python logger bridge initialized
/Users/sigvardbratlie/Documents/Projects/master-thesis/.venv/lib/python3.13/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "output_schema" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]
/Users/sigvardbratlie/Documents/Projects/master-thesis/.venv/lib/python3.13/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "stream" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]
INFO:evals.dataset:Found 18 files under datasets/test/01_data/
INFO:evals.dataset:Session Prosjekt-initialisering | – → 2020-03-15 | 18 candidates, 18 new
INFO:evals.dataset:Found 18 files under datasets/test/01_data/
INFO:evals.dataset:Session Prosjekt-initialisering | – → 2020-03-15 | 18 attachments (cumulative)
INFO:__main__:Done —

In [3]:
ds_custom.data

{'project_id': 'b3f45644-6222-4593-8955-cf4d8e0b00d5',
 'dataset_name': 'test',
 'user_id': '53d63d18-cfa1-416e-96e8-770c8f66507b',
 'last_updated': '2026-02-25T12:55:16.732547+01:00',
 'sessions': [{'session': 0,
   'date': '2020-03-15',
   'session_id': '3eb54a6b-ca3d-4d21-9ea1-b48f4e0228e1',
   'session_name': 'Prosjekt-initialisering',
   'init_query': 'Jeg er advokat og representerer kjøperparet Anders og Berit Kristiansen i en eiendomskjøpssak. De kjøpte en eiendom på Fjellveien 42A i Stavanger kommune den 1. juni 2019, med overtakelse 1. august 2019. \nVi er nå i mars 2020 og det har dukket opp flere problemer med eiendommen.  Jeg trenger din hjelp til å organisere saksinnholdet, identifisere de juridiske problemstillingene, og vurdere mulige tiltak.',
   'conversation': [{'input': 'Gi meg en kort og konsis oppsummering av sakens faktiske bakgrunn og utvikling så langt, basert på dokumentene jeg har lastet opp. \nFokuser på de viktigste hendelsene og problemstillingene. Hva er k

In [4]:
ds_baseline.data

{'project_id': 'b3f45644-6222-4593-8955-cf4d8e0b00d5',
 'dataset_name': 'test',
 'user_id': '53d63d18-cfa1-416e-96e8-770c8f66507b',
 'last_updated': '2026-02-25T12:55:16.732547+01:00',
 'sessions': [{'session': 0,
   'date': '2020-03-15',
   'session_id': '3eb54a6b-ca3d-4d21-9ea1-b48f4e0228e1',
   'session_name': 'Prosjekt-initialisering',
   'init_query': 'Jeg er advokat og representerer kjøperparet Anders og Berit Kristiansen i en eiendomskjøpssak. De kjøpte en eiendom på Fjellveien 42A i Stavanger kommune den 1. juni 2019, med overtakelse 1. august 2019. \nVi er nå i mars 2020 og det har dukket opp flere problemer med eiendommen.  Jeg trenger din hjelp til å organisere saksinnholdet, identifisere de juridiske problemstillingene, og vurdere mulige tiltak.',
   'conversation': [{'input': 'Gi meg en kort og konsis oppsummering av sakens faktiske bakgrunn og utvikling så langt, basert på dokumentene jeg har lastet opp. \nFokuser på de viktigste hendelsene og problemstillingene. Hva er k

### GATHER RESULTS

In [5]:
from evals import CollectAgentResult

In [ ]:
car_custom = CollectAgentResult(data_custom, llm_model="google_gemini-2.5-flash", custom_agent=True)
await car_custom.run_agent(embed_to_vectorstore=False, save_to_storage=False)

car_baseline = CollectAgentResult(data_baseline, llm_model="google_gemini-2.5-flash", custom_agent=False)
await car_baseline.run_agent(embed_to_vectorstore=False, save_to_storage=False)

INFO:evals.dataset:Agent initialized with AsyncPostgresSaver checkpointer
INFO:evals.dataset:=========== STARTING EVALUATION ===========
INFO:evals.dataset:Dataset: test | Sessions: 1 | Project: b3f45644-6222-4593-8955-cf4d8e0b00d5 | User: 53d63d18-cfa1-416e-96e8-770c8f66507b
INFO:evals.dataset:Session 0 | 2020-03-15 | Prosjekt-initialisering | 18 attachments
INFO:agent.context_manager:==== ATTACHMENT ELEMENT DEBUG == 
{'description': 'Damage report from Takst & Analyse AS for Anders and Berit Kristiansen regarding a property at Fjellveien 42A. It details a leakage through the concrete deck caused by column shoes puncturing the membrane. Recommends further investigation and repair of the leakage.', 'significance': 'high', 'party_roles': ['Anders Kristiansen (plaintiff)', 'Berit Kristiansen (plaintiff)', 'Takst & Analyse AS (expert)'], 'deadlines': [], 'damages': [{'damage_id': None, 'category': 'direct_losses', 'amount': None, 'basis': 'Lekkasje via betongdekket som utgjør taket på utl

CancelledError: 

INFO:agent.context_manager:==== ATTACHMENT ELEMENT DEBUG == 
{'description': 'Documentation of moving and cleaning costs incurred by Anders and Berit Kristiansen for their property at Fjellveien 42A, including past expenses and future estimates.', 'significance': 'medium', 'party_roles': ['plaintiff'], 'deadlines': [], 'damages': [{'damage_id': None, 'category': 'consequential', 'amount': 54800, 'basis': 'Documented moving and cleaning costs for property Fjellveien 42A.', 'supporting_evidence': ['31980836-46d1-44a9-b8e9-7998154f0fb8'], 'file_id': None, 'email_id': None, 'party_role': 'plaintiff'}, {'damage_id': None, 'category': 'consequential', 'amount': 50000, 'basis': 'Estimated future moving costs based on offers from moving companies.', 'supporting_evidence': ['31980836-46d1-44a9-b8e9-7998154f0fb8'], 'file_id': None, 'email_id': None, 'party_role': 'plaintiff'}], 'claims': [], 'file_id': '31980836-46d1-44a9-b8e9-7998154f0fb8', 'key_provisions': [], 'file_date': '2023-01-20', 'cate

In [ ]:
from langsmith import Client as LangSmithClient
import os

def get_token_counts(eval_run_id: str) -> dict:
    """Query LangSmith for total token usage across an eval run."""
    client = LangSmithClient()
    project_name = os.getenv("LANGCHAIN_PROJECT", "default")
    runs = list(client.list_runs(
        project_name=project_name,
        filter=f'has(tags, "{eval_run_id}")',
        run_type="llm",
    ))
    return {
        "eval_run_id": eval_run_id,
        "input_tokens": sum(r.prompt_tokens or 0 for r in runs),
        "output_tokens": sum(r.completion_tokens or 0 for r in runs),
        "total_tokens": sum(r.total_tokens or 0 for r in runs),
        "llm_calls": len(runs),
    }

token_counts_custom = get_token_counts(car_custom.data["eval_run_id"])
car_custom.data["token_counts"] = token_counts_custom   
ds_custom.save_results(car_custom.data)
print(token_counts_custom)

{'eval_run_id': 'f15d86ea-48f3-4bfb-8150-08257801837c', 'input_tokens': 18165, 'output_tokens': 22737, 'total_tokens': 40902, 'llm_calls': 4}


In [ ]:
token_counts_baseline = get_token_counts(car_baseline.data["eval_run_id"])
car_baseline.data["token_counts"] = token_counts_baseline
ds_custom.save_results(car_baseline.data)
print(token_counts_baseline)

INFO:evals.dataset:Results saved to datasets/THRD-2021-163881/04_results/google_gemini-2.5-flash_custom_2026-02-25_13-33-56.json


### EVALUATE RESULTS

In [11]:
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval
from deepeval import evaluate
from deepeval.test_case import ConversationalTestCase, Turn, TurnParams
from deepeval.metrics import ConversationalGEval

In [ ]:
from evals import Dataset
dataset_name = "test"
ds = Dataset(dataset_name)
results = ds.load_results()
result = results.get("datasets/test/04_results/google_gemini-2.5-flash_2026-02-24_13-41-07.json")
session1 = result["sessions"][0]
conv1 = session1["conversation"][0]
conv1

{'input': 'Gi meg en kort og konsis oppsummering av sakens faktiske bakgrunn og utvikling så langt, basert på dokumentene jeg har lastet opp. \nFokuser på de viktigste hendelsene og problemstillingene. Hva er kjernen i saken?',
 'answer': "Anders og Berit Kristiansen kjøpte en eiendom på Fjellveien 42A i Stavanger kommune den 1. juni 2019, med overtakelse 1. august 2019.\n\nSelger meldte fra om vannlekkasje (juli 2019), altså før overtagelse, hvor lekkasjen skal ha blitt forsøkt utbedret i forkant av overtagelsen. \nKun 18 dager etter overtagelsen var lekkasjen tilbake og dette ble meldt fra om av kjøper til selgeren pr epost (18.aug)\nDet ble utarbeidet en skaderapporter (10.sept 2019) som tar for seg vannlekkasjen og de omkringliggende bygningsdelene. Rapporten konkluderer med at membranen var punktert av søyleskoene og at det er årsaken til vanngjennomtrengningen.\n\nSiste oppdatering er fra mars 2020 hvor det ble laget en rapport på betongdekket over utleiedelen for å avklare om de

In [ ]:
from deepeval.evaluate.types import EvaluationResult
from google.cloud import storage
import json
class Evaluater:
    def __init__(self, client=None, bucket_name="master-thesis-prod"):
        self._client = client or storage.Client()
        self.bucket = self._client.bucket(bucket_name)

    def collect_single(self, conversation_turn : dict,session_name : str = "unknown"): 
        if "input" not in conversation_turn or "model_response" not in conversation_turn or "answer" not in conversation_turn:
            raise ValueError("Conversation turn must contain 'input', 'model_response', and 'answer' fields.")
        if not conversation_turn.get("input") or not conversation_turn.get("model_response") or not conversation_turn.get("answer"):
            logger.warning("Conversation turn is missing required fields. Skipping evaluation for this turn.")
            return
        test_case = LLMTestCase(name = f"Turn {conversation_turn.get('order', 'unknown')} in session {session_name}",
                                input = conversation_turn.get("input"), 
                                actual_output = conversation_turn.get("model_response"),
                                expected_output=conversation_turn.get("answer"),
                                additional_metadata={"turn_order": conversation_turn.get("order", "unknown"),
                                                     "query_id" : conversation_turn.get("query_id", "unknown")})

        correctness = GEval(name = "correctness",
                            criteria="Determine if actual_output is factually correct based on expected_output",
                            evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
                            threshold=0.5)

        return {"test_case": test_case, "metric": correctness}

    def eval_conversation(self, conversation : list[dict]):
        turns = []
        for item in conversation:
            turns.append(Turn(role="user", content=item["input"]))
            turns.append(Turn(role="assistant", content=item["model_response"]))

        convo_test_case = ConversationalTestCase(turns=turns)

        metric = ConversationalGEval(
            name="Legal Accuracy",
            criteria="Evaluate whether the assistant's legal analysis is accurate and consistent across the conversation.",
            evaluation_params=[TurnParams.CONTENT],
            threshold=0.5
        )

        return evaluate(test_cases=[convo_test_case], metrics=[metric])
    
    def run_session_eval(self, session : dict):
        test_cases = []
        metrics = []
        for conversation_turn in session.get("conversation", []):
            collected = self.collect_single(conversation_turn=conversation_turn, session_name=session.get("session_name", "unknown"))
            if collected:
                test_cases.append(collected["test_case"])
                metrics.append(collected["metric"])

        return evaluate(test_cases=test_cases, metrics=metrics)

    def run_evaluation(self):
        sessions = self.data.get("sessions", [])
        for session in sessions:
            self.run_session_eval(session)

    def save_evaluation_results(self, results : list[EvaluationResult]):
        output = {
            "dataset_name": self.data.get("dataset_name"),
            "project_id": self.data.get("project"),
            "user_id": self.data.get("user"),
            "eval_run_id": self.data.get("eval_run_id"),
            "llm_model": self.data.get("llm_model"),
            "custom_agent": self.data.get("custom_agent"),
            "results": [r.model_dump() for r in results if r and isinstance(r, EvaluationResult)],
            }
        agent_type = "custom" if self.data.get("custom_agent") else "baseline"
        filepath = f'datasets/{self.data.get("dataset_name")}/05_evals/{self.data.get("llm_model")}_{agent_type}_{self.data.get("eval_run_id")}.json'
        blob = self.bucket.blob(filepath)
        blob.upload_from_string(json.dumps(output), content_type='application/json')

In [19]:
e = Evaluater()

In [27]:
results[0].model_dump()

{'test_results': [{'name': 'test_case_0',
   'success': True,
   'metrics_data': [{'name': 'correctness [GEval]',
     'threshold': 0.5,
     'success': True,
     'score': 0.9025964279761473,
     'reason': 'The actual output provides a detailed and accurate summary of the factual background and development, closely matching the expected output. It covers the key events: the property purchase, the initial and recurring water leak, the timeline of reports and findings, and the discovery of construction defects. It also correctly identifies the core issue as relating to the water leak and construction faults. The only minor shortcoming is that the actual output includes some additional legal context and details about rental consequences and costs, which, while accurate, go slightly beyond the strict factual summary requested. However, there are no factual inaccuracies or significant omissions.',
     'strict_mode': False,
     'evaluation_model': 'gpt-4.1',
     'error': None,
     'eva

In [20]:
results = e.run_session_eval(session1)

✨ You're running DeepEval's latest correctness [GEval] Metric! (using gpt-4.1, strict=False, async_mode=True)...

Output()

INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases




Metrics Summary

  - ✅ correctness [GEval] (score: 0.9025964279761473, threshold: 0.5, strict: False, evaluation model: gpt-4.1, reason: The actual output provides a detailed and accurate summary of the factual background and development, closely matching the expected output. It covers the key events: the property purchase, the initial and recurring water leak, the timeline of reports and findings, and the discovery of construction defects. It also correctly identifies the core issue as relating to the water leak and construction faults. The only minor shortcoming is that the actual output includes some additional legal context and details about rental consequences and costs, which, while accurate, go slightly beyond the strict factual summary requested. However, there are no factual inaccuracies or significant omissions., error: None)

For test case:

  - input: Gi meg en kort og konsis oppsummering av sakens faktiske bakgrunn og utvikling så langt, basert på dokumentene jeg har las

⚠ WARNING: No hyperparameters logged.
» ]8;id=720845;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

...

Output()

INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases




Metrics Summary

  - ✅ correctness [GEval] (score: 0.75244787883822, threshold: 0.5, strict: False, evaluation model: gpt-4.1, reason: The actual output aligns well with the expected output by clearly stating that the salgsoppgaven does not describe the problematic conditions and even highlights that it contains misleading or incorrect information regarding the issues later discovered. It provides additional factual context about the case, referencing specific details about the lack of mention of leaks or construction issues in the sales prospectus. However, it goes beyond the expected output by including extensive background and legal context, which, while accurate, is not strictly necessary for answering the question. There are no factual inaccuracies or omissions regarding the core issue, but the answer could be more concise and focused., error: None)

For test case:

  - input: Beskriver salgsoppgaven noen av de forhold som nå er problematiske?
  - actual output: Saken dreier seg

⚠ WARNING: No hyperparameters logged.
» ]8;id=218820;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=868189;https://app.confident-ai.com/project/cmlt7bho400hsmx1e0kg8cu1y/test-runs/cmm1yb40k000rqx1e80e9ptka/regression-testing\https://app.confident-ai.com/project/cmlt7bho400hsmx1e0kg8cu1y/test-runs/cmm1yb40k000rqx1e80e9ptka/regression-testi]8;;\
]8;id=868189;https://app.confident-ai.com/project/cmlt7bho400hsmx1e0kg8cu1y/test-runs/cmm1yb40k000rqx1e80e9ptka/regression-testing\ng]8;;\

In [24]:
results

[EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='correctness [GEval]', threshold=0.5, success=True, score=0.9025964279761473, reason='The actual output provides a detailed and accurate summary of the factual background and development, closely matching the expected output. It covers the key events: the property purchase, the initial and recurring water leak, the timeline of reports and findings, and the discovery of construction defects. It also correctly identifies the core issue as relating to the water leak and construction faults. The only minor shortcoming is that the actual output includes some additional legal context and details about rental consequences and costs, which, while accurate, go slightly beyond the strict factual summary requested. However, there are no factual inaccuracies or significant omissions.', strict_mode=False, evaluation_model='gpt-4.1', error=None, evaluation_cost=0.00597, verbose_logs='Criteria:\